Pyspark capstone project
# Abhishek_jadhav_69137
# ETL pipeline

In [58]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Retail Data Engineering Pipeline") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("Application:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)

Spark Version: 3.5.9
Application: Retail Data Engineering Pipeline
Master: local[*]


In [59]:
customers = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/customers.csv")

print(" CUSTOMER DATA ")
customers.show()
customers.printSchema()

 CUSTOMER DATA 
+-----------+---------------+------+---+---------+-----------------+
|customer_id|  customer_name|gender|age|     city|registration_date|
+-----------+---------------+------+---+---------+-----------------+
|       C101|Abhishek Jadhav|  Male| 26|   Mumbai|       2025-01-15|
|       C102|     Tony Stark|  male| 25|     Pune|       2025-02-10|
|       C103| Amitabh Sharma|  Male| 48|    Delhi|       2025-01-20|
|       C104|     Jaya Mehta|Female| 42|Ahmedabad|       2025-03-12|
|       C105|    Salman Khan|  Male| 35|   Mumbai|       2025-04-05|
|       C106|    Rahul Verma|  Male| 29|     Pune|       2025-05-18|
|       C107|     Priya Shah|Female| 31|Ahmedabad|       2025-02-25|
|       C108|   Kapil Sharma|  Male| 39|    Delhi|       2025-06-10|
|       C109|     Neha Singh|Female| 28|   Mumbai|       2025-07-01|
|       C110|    Samay Raina|  Male| 27|     Pune|       2025-07-15|
|       C103| Amitabh Sharma|  Male| 48|    Delhi|       2025-01-20|
|       C111|     

In [60]:
products = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/products.csv")

print(" PRODUCT DATA ")
products.show()
products.printSchema()

 PRODUCT DATA 
+----------+-------------+-----------+-------+----------+
|product_id| product_name|   category|  brand|unit_price|
+----------+-------------+-----------+-------+----------+
|      P101|       Laptop|Electronics|   Dell|     65000|
|      P102|   Smartphone|Electronics|Samsung|     45000|
|      P103|   Headphones|Electronics|   Sony|      5000|
|      P104| Office Chair|  Furniture|   Ikea|     12000|
|      P105|  Study Table|  Furniture|   Ikea|     15000|
|      P106|Running Shoes|   Footwear|   Nike|      7000|
|      P107| Sports Shoes|   Footwear| Adidas|      6500|
|      P108|      T-Shirt|   Clothing|   Puma|      1500|
|      P109|       Jacket|   Clothing|  Levis|      4500|
|      P110|  Smart Watch|Electronics|  Apple|     30000|
|      P106|Running Shoes|   Footwear|   Nike|      7000|
|      P111|         NULL|   Clothing|   Puma|      2000|
+----------+-------------+-----------+-------+----------+

root
 |-- product_id: string (nullable = true)
 |-- prod

In [61]:
sales = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/sales.csv")

print(" SALES DATA ")
sales.show()
sales.printSchema()

 SALES DATA 
+--------------+-----------+----------+-------------+----------------+------------+
|transaction_id|customer_id|product_id|quantity_sold|transaction_date|sales_amount|
+--------------+-----------+----------+-------------+----------------+------------+
|         T1001|       C101|      P101|            1|      2026-01-10|       65000|
|         T1002|       C102|      P102|            2|      2026-01-15|       90000|
|         T1003|       C103|      P103|            3|      2026-02-05|       15000|
|         T1004|       C101|      P104|            2|      2026-02-12|       24000|
|         T1005|       C104|      P105|            1|      2026-03-08|       15000|
|         T1006|       C105|      P102|            1|      2026-03-15|       45000|
|         T1007|       C106|      P106|            2|      2026-04-05|       14000|
|         T1008|       C107|      P107|            1|      2026-04-18|        6500|
|         T1009|       C108|      P108|            4|      2026

In [62]:
print("Customer Records:", customers.count())
print("Product Records:", products.count())
print("Sales Records:", sales.count())

Customer Records: 12
Product Records: 12
Sales Records: 20


In [63]:
print("  CUSTOMER NULL VALUES  ")

customers.select([
    count(
        when(col(column).isNull(), column)
    ).alias(column)
    for column in customers.columns
]).show()

  CUSTOMER NULL VALUES  
+-----------+-------------+------+---+----+-----------------+
|customer_id|customer_name|gender|age|city|registration_date|
+-----------+-------------+------+---+----+-----------------+
|          0|            1|     0|  0|   0|                0|
+-----------+-------------+------+---+----+-----------------+



In [64]:
print("  PRODUCT NULL VALUES  ")

products.select([
    count(
        when(col(column).isNull(), column)
    ).alias(column)
    for column in products.columns
]).show()

  PRODUCT NULL VALUES  
+----------+------------+--------+-----+----------+
|product_id|product_name|category|brand|unit_price|
+----------+------------+--------+-----+----------+
|         0|           1|       0|    0|         0|
+----------+------------+--------+-----+----------+



In [65]:
print("  SALES NULL VALUES  ")

sales.select([
    count(
        when(col(column).isNull(), column)
    ).alias(column)
    for column in sales.columns
]).show()

  SALES NULL VALUES  
+--------------+-----------+----------+-------------+----------------+------------+
|transaction_id|customer_id|product_id|quantity_sold|transaction_date|sales_amount|
+--------------+-----------+----------+-------------+----------------+------------+
|             0|          1|         1|            0|               0|           0|
+--------------+-----------+----------+-------------+----------------+------------+



In [66]:
print("  DUPLICATE CUSTOMERS  ")

customers.groupBy("customer_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

  DUPLICATE CUSTOMERS  
+-----------+-----+
|customer_id|count|
+-----------+-----+
|       C103|    2|
+-----------+-----+



In [67]:
print("  DUPLICATE PRODUCTS  ")

products.groupBy("product_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

  DUPLICATE PRODUCTS  
+----------+-----+
|product_id|count|
+----------+-----+
|      P106|    2|
+----------+-----+



In [68]:
print("  DUPLICATE TRANSACTIONS  ")

sales.groupBy("transaction_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

  DUPLICATE TRANSACTIONS  
+--------------+-----+
|transaction_id|count|
+--------------+-----+
|         T1016|    2|
+--------------+-----+



In [69]:
clean_customers = customers \
    .dropDuplicates(["customer_id"]) \
    .dropna(subset=["customer_id", "customer_name"]) \
    .withColumn(
        "customer_name",
        trim(col("customer_name"))
    ) \
    .withColumn(
        "gender",
        trim(col("gender"))
    ) \
    .withColumn(
        "city",
        trim(col("city"))
    )

print("  CLEANED CUSTOMER DATA  ")
clean_customers.show()

  CLEANED CUSTOMER DATA  
+-----------+---------------+------+---+---------+-----------------+
|customer_id|  customer_name|gender|age|     city|registration_date|
+-----------+---------------+------+---+---------+-----------------+
|       C101|Abhishek Jadhav|  Male| 26|   Mumbai|       2025-01-15|
|       C102|     Tony Stark|  male| 25|     Pune|       2025-02-10|
|       C103| Amitabh Sharma|  Male| 48|    Delhi|       2025-01-20|
|       C104|     Jaya Mehta|Female| 42|Ahmedabad|       2025-03-12|
|       C105|    Salman Khan|  Male| 35|   Mumbai|       2025-04-05|
|       C106|    Rahul Verma|  Male| 29|     Pune|       2025-05-18|
|       C107|     Priya Shah|Female| 31|Ahmedabad|       2025-02-25|
|       C108|   Kapil Sharma|  Male| 39|    Delhi|       2025-06-10|
|       C109|     Neha Singh|Female| 28|   Mumbai|       2025-07-01|
|       C110|    Samay Raina|  Male| 27|     Pune|       2025-07-15|
+-----------+---------------+------+---+---------+-----------------+



In [70]:
clean_products = products \
    .dropDuplicates(["product_id"]) \
    .dropna(subset=[
        "product_id",
        "product_name",
        "category",
        "unit_price"
    ]) \
    .filter(
        col("unit_price") > 0
    ) \
    .withColumn(
        "product_name",
        trim(col("product_name"))
    ) \
    .withColumn(
        "category",
        trim(col("category"))
    ) \
    .withColumn(
        "brand",
        trim(col("brand"))
    )

print("  CLEANED PRODUCT DATA  ")
clean_products.show()

  CLEANED PRODUCT DATA  
+----------+-------------+-----------+-------+----------+
|product_id| product_name|   category|  brand|unit_price|
+----------+-------------+-----------+-------+----------+
|      P101|       Laptop|Electronics|   Dell|     65000|
|      P102|   Smartphone|Electronics|Samsung|     45000|
|      P103|   Headphones|Electronics|   Sony|      5000|
|      P104| Office Chair|  Furniture|   Ikea|     12000|
|      P105|  Study Table|  Furniture|   Ikea|     15000|
|      P106|Running Shoes|   Footwear|   Nike|      7000|
|      P107| Sports Shoes|   Footwear| Adidas|      6500|
|      P108|      T-Shirt|   Clothing|   Puma|      1500|
|      P109|       Jacket|   Clothing|  Levis|      4500|
|      P110|  Smart Watch|Electronics|  Apple|     30000|
+----------+-------------+-----------+-------+----------+



In [71]:
clean_sales = sales \
    .dropDuplicates(["transaction_id"]) \
    .dropna(subset=[
        "transaction_id",
        "customer_id",
        "product_id",
        "quantity_sold",
        "transaction_date"
    ]) \
    .filter(
        col("quantity_sold") > 0
    )

print(" CLEAN SALES DATA  ")
clean_sales.show()

 CLEAN SALES DATA  
+--------------+-----------+----------+-------------+----------------+------------+
|transaction_id|customer_id|product_id|quantity_sold|transaction_date|sales_amount|
+--------------+-----------+----------+-------------+----------------+------------+
|         T1001|       C101|      P101|            1|      2026-01-10|       65000|
|         T1002|       C102|      P102|            2|      2026-01-15|       90000|
|         T1003|       C103|      P103|            3|      2026-02-05|       15000|
|         T1004|       C101|      P104|            2|      2026-02-12|       24000|
|         T1005|       C104|      P105|            1|      2026-03-08|       15000|
|         T1006|       C105|      P102|            1|      2026-03-15|       45000|
|         T1007|       C106|      P106|            2|      2026-04-05|       14000|
|         T1008|       C107|      P107|            1|      2026-04-18|        6500|
|         T1009|       C108|      P108|            4|   

In [72]:
print("Clean Customer Records:", clean_customers.count())
print("Clean Product Records:", clean_products.count())
print("Clean Sales Records:", clean_sales.count())

Clean Customer Records: 10
Clean Product Records: 10
Clean Sales Records: 16


In [73]:
clean_customers = clean_customers.cache()
clean_products = clean_products.cache()

clean_customers.count()
clean_products.count()

print("Customer Cached:", clean_customers.is_cached)
print("Product Cached:", clean_products.is_cached)

Customer Cached: True
Product Cached: True


In [74]:
sales_customer = clean_sales.join(
    broadcast(clean_customers),
    clean_sales.customer_id == clean_customers.customer_id,
    "inner"
).drop(clean_customers.customer_id)

print(" SALE + CUSTOMER  ")
sales_customer.show()

 SALE + CUSTOMER  
+--------------+-----------+----------+-------------+----------------+------------+---------------+------+---+---------+-----------------+
|transaction_id|customer_id|product_id|quantity_sold|transaction_date|sales_amount|  customer_name|gender|age|     city|registration_date|
+--------------+-----------+----------+-------------+----------------+------------+---------------+------+---+---------+-----------------+
|         T1001|       C101|      P101|            1|      2026-01-10|       65000|Abhishek Jadhav|  Male| 26|   Mumbai|       2025-01-15|
|         T1002|       C102|      P102|            2|      2026-01-15|       90000|     Tony Stark|  male| 25|     Pune|       2025-02-10|
|         T1003|       C103|      P103|            3|      2026-02-05|       15000| Amitabh Sharma|  Male| 48|    Delhi|       2025-01-20|
|         T1004|       C101|      P104|            2|      2026-02-12|       24000|Abhishek Jadhav|  Male| 26|   Mumbai|       2025-01-15|
|       

In [75]:
retail_data = sales_customer.join(
    broadcast(clean_products),
    sales_customer.product_id == clean_products.product_id,
    "inner"
).drop(clean_products.product_id)

print("  COMBINED RETAIL DATA  ")
retail_data.show()

  COMBINED RETAIL DATA  
+--------------+-----------+----------+-------------+----------------+------------+---------------+------+---+---------+-----------------+-------------+-----------+-------+----------+
|transaction_id|customer_id|product_id|quantity_sold|transaction_date|sales_amount|  customer_name|gender|age|     city|registration_date| product_name|   category|  brand|unit_price|
+--------------+-----------+----------+-------------+----------------+------------+---------------+------+---+---------+-----------------+-------------+-----------+-------+----------+
|         T1001|       C101|      P101|            1|      2026-01-10|       65000|Abhishek Jadhav|  Male| 26|   Mumbai|       2025-01-15|       Laptop|Electronics|   Dell|     65000|
|         T1002|       C102|      P102|            2|      2026-01-15|       90000|     Tony Stark|  male| 25|     Pune|       2025-02-10|   Smartphone|Electronics|Samsung|     45000|
|         T1003|       C103|      P103|            3|  

In [76]:
retail_data = retail_data.withColumn(
    "total_revenue",
    col("quantity_sold") * col("unit_price")
)

print("  RETAIL DATA WITH REVENUE  ")

retail_data.select(
    "transaction_id",
    "customer_name",
    "product_name",
    "quantity_sold",
    "unit_price",
    "total_revenue"
).show()

  RETAIL DATA WITH REVENUE  
+--------------+---------------+-------------+-------------+----------+-------------+
|transaction_id|  customer_name| product_name|quantity_sold|unit_price|total_revenue|
+--------------+---------------+-------------+-------------+----------+-------------+
|         T1001|Abhishek Jadhav|       Laptop|            1|     65000|        65000|
|         T1002|     Tony Stark|   Smartphone|            2|     45000|        90000|
|         T1003| Amitabh Sharma|   Headphones|            3|      5000|        15000|
|         T1004|Abhishek Jadhav| Office Chair|            2|     12000|        24000|
|         T1005|     Jaya Mehta|  Study Table|            1|     15000|        15000|
|         T1006|    Salman Khan|   Smartphone|            1|     45000|        45000|
|         T1007|    Rahul Verma|Running Shoes|            2|      7000|        14000|
|         T1008|     Priya Shah| Sports Shoes|            1|      6500|         6500|
|         T1009|   Kapil 

In [77]:
retail_data = retail_data \
    .withColumn(
        "transaction_year",
        year(col("transaction_date"))
    ) \
    .withColumn(
        "transaction_month",
        month(col("transaction_date"))
    )

retail_data.select(
    "transaction_id",
    "transaction_date",
    "transaction_year",
    "transaction_month"
).show()

+--------------+----------------+----------------+-----------------+
|transaction_id|transaction_date|transaction_year|transaction_month|
+--------------+----------------+----------------+-----------------+
|         T1001|      2026-01-10|            2026|                1|
|         T1002|      2026-01-15|            2026|                1|
|         T1003|      2026-02-05|            2026|                2|
|         T1004|      2026-02-12|            2026|                2|
|         T1005|      2026-03-08|            2026|                3|
|         T1006|      2026-03-15|            2026|                3|
|         T1007|      2026-04-05|            2026|                4|
|         T1008|      2026-04-18|            2026|                4|
|         T1009|      2026-05-11|            2026|                5|
|         T1010|      2026-05-20|            2026|                5|
|         T1011|      2026-06-05|            2026|                6|
|         T1012|      2026-06-18| 

In [78]:
category_sales = retail_data.groupBy(
    "category"
).agg(
    sum("total_revenue").alias("category_revenue")
).orderBy(
    col("category_revenue").desc()
)

print("  SALES BY PRODUCT CATEGORY  ")
category_sales.show()

  SALES BY PRODUCT CATEGORY  
+-----------+----------------+
|   category|category_revenue|
+-----------+----------------+
|Electronics|          365000|
|   Footwear|           41500|
|  Furniture|           39000|
|   Clothing|           22500|
+-----------+----------------+



In [79]:
city_sales = retail_data.groupBy(
    "city"
).agg(
    sum("total_revenue").alias("city_revenue")
).orderBy(
    col("city_revenue").desc()
)

print("  SALES BY CITY  ")
city_sales.show()

  SALES BY CITY  
+---------+------------+
|     city|city_revenue|
+---------+------------+
|   Mumbai|      201500|
|     Pune|      179000|
|    Delhi|       66000|
|Ahmedabad|       21500|
+---------+------------+



In [80]:
customer_revenue = retail_data.groupBy(
    "customer_id",
    "customer_name"
).agg(
    sum("total_revenue").alias("customer_revenue")
).orderBy(
    col("customer_revenue").desc()
)

print("  CUSTOMER WISE REVENUE ")
customer_revenue.show()

  CUSTOMER WISE REVENUE 
+-----------+---------------+----------------+
|customer_id|  customer_name|customer_revenue|
+-----------+---------------+----------------+
|       C102|     Tony Stark|          155000|
|       C101|Abhishek Jadhav|          126500|
|       C105|    Salman Khan|           66000|
|       C103| Amitabh Sharma|           60000|
|       C104|     Jaya Mehta|           15000|
|       C106|    Rahul Verma|           14000|
|       C110|    Samay Raina|           10000|
|       C109|     Neha Singh|            9000|
|       C107|     Priya Shah|            6500|
|       C108|   Kapil Sharma|            6000|
+-----------+---------------+----------------+



In [81]:
customer_frequency = retail_data.groupBy(
    "customer_id",
    "customer_name"
).agg(
    count("transaction_id").alias("purchase_frequency")
).orderBy(
    col("purchase_frequency").desc()
)

print(" CUSTOMER PURCHASE FREQUENCY  ")
customer_frequency.show()

 CUSTOMER PURCHASE FREQUENCY  
+-----------+---------------+------------------+
|customer_id|  customer_name|purchase_frequency|
+-----------+---------------+------------------+
|       C101|Abhishek Jadhav|                 4|
|       C105|    Salman Khan|                 2|
|       C103| Amitabh Sharma|                 2|
|       C102|     Tony Stark|                 2|
|       C110|    Samay Raina|                 1|
|       C107|     Priya Shah|                 1|
|       C104|     Jaya Mehta|                 1|
|       C108|   Kapil Sharma|                 1|
|       C109|     Neha Singh|                 1|
|       C106|    Rahul Verma|                 1|
+-----------+---------------+------------------+



In [82]:
high_value_customers = customer_revenue.withColumn(
    "customer_category",
    when(
        col("customer_revenue") >= 50000,
        "High Value"
    ).otherwise("Regular")
)

print("  HIGH-VALUE CUSTOMERS  ")
high_value_customers.show()

  HIGH-VALUE CUSTOMERS  
+-----------+---------------+----------------+-----------------+
|customer_id|  customer_name|customer_revenue|customer_category|
+-----------+---------------+----------------+-----------------+
|       C102|     Tony Stark|          155000|       High Value|
|       C101|Abhishek Jadhav|          126500|       High Value|
|       C105|    Salman Khan|           66000|       High Value|
|       C103| Amitabh Sharma|           60000|       High Value|
|       C104|     Jaya Mehta|           15000|          Regular|
|       C106|    Rahul Verma|           14000|          Regular|
|       C110|    Samay Raina|           10000|          Regular|
|       C109|     Neha Singh|            9000|          Regular|
|       C107|     Priya Shah|            6500|          Regular|
|       C108|   Kapil Sharma|            6000|          Regular|
+-----------+---------------+----------------+-----------------+



In [83]:
product_revenue = retail_data.groupBy(
    "product_id",
    "product_name",
    "category"
).agg(
    sum("total_revenue").alias("product_revenue")
).orderBy(
    col("product_revenue").desc()
)

print("  PRODUCT REVENUE  ")
product_revenue.show()

  PRODUCT REVENUE  
+----------+-------------+-----------+---------------+
|product_id| product_name|   category|product_revenue|
+----------+-------------+-----------+---------------+
|      P102|   Smartphone|Electronics|         180000|
|      P101|       Laptop|Electronics|         130000|
|      P106|Running Shoes|   Footwear|          35000|
|      P110|  Smart Watch|Electronics|          30000|
|      P103|   Headphones|Electronics|          25000|
|      P104| Office Chair|  Furniture|          24000|
|      P105|  Study Table|  Furniture|          15000|
|      P108|      T-Shirt|   Clothing|          13500|
|      P109|       Jacket|   Clothing|           9000|
|      P107| Sports Shoes|   Footwear|           6500|
+----------+-------------+-----------+---------------+



In [84]:
product_revenue_category = product_revenue.withColumn(
    "revenue_category",
    when(
        col("product_revenue") >= 50000,
        "High Contribution"
    ).when(
        col("product_revenue") >= 20000,
        "Medium Contribution"
    ).otherwise(
        "Low Contribution"
    )
)

print("  PRODUCT REVENUE CONTRIBUTION  ")
product_revenue_category.show()

  PRODUCT REVENUE CONTRIBUTION  
+----------+-------------+-----------+---------------+-------------------+
|product_id| product_name|   category|product_revenue|   revenue_category|
+----------+-------------+-----------+---------------+-------------------+
|      P102|   Smartphone|Electronics|         180000|  High Contribution|
|      P101|       Laptop|Electronics|         130000|  High Contribution|
|      P106|Running Shoes|   Footwear|          35000|Medium Contribution|
|      P110|  Smart Watch|Electronics|          30000|Medium Contribution|
|      P103|   Headphones|Electronics|          25000|Medium Contribution|
|      P104| Office Chair|  Furniture|          24000|Medium Contribution|
|      P105|  Study Table|  Furniture|          15000|   Low Contribution|
|      P108|      T-Shirt|   Clothing|          13500|   Low Contribution|
|      P109|       Jacket|   Clothing|           9000|   Low Contribution|
|      P107| Sports Shoes|   Footwear|           6500|   Low Contri

In [85]:
monthly_revenue = retail_data.groupBy(
    "transaction_year",
    "transaction_month"
).agg(
    sum("total_revenue").alias("monthly_revenue")
).orderBy(
    "transaction_year",
    "transaction_month"
)

print("  MONTHLY REVENUE  ")
monthly_revenue.show()

  MONTHLY REVENUE  
+----------------+-----------------+---------------+
|transaction_year|transaction_month|monthly_revenue|
+----------------+-----------------+---------------+
|            2026|                1|         155000|
|            2026|                2|          39000|
|            2026|                3|          60000|
|            2026|                4|          20500|
|            2026|                5|          36000|
|            2026|                6|          74000|
|            2026|                7|          31000|
|            2026|                8|          52500|
+----------------+-----------------+---------------+



In [86]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

month_window = Window.orderBy(
    "transaction_year",
    "transaction_month"
)

monthly_growth = monthly_revenue.withColumn(
    "previous_month_revenue",
    lag("monthly_revenue").over(month_window)
).withColumn(
    "monthly_growth_percent",
    round(
        (
            (
                col("monthly_revenue")
                - col("previous_month_revenue")
            )
            / col("previous_month_revenue")
        ) * 100,
        2
    )
)

print("  MONTHLY SALES GROWTH  ")
monthly_growth.show()

  MONTHLY SALES GROWTH  
+----------------+-----------------+---------------+----------------------+----------------------+
|transaction_year|transaction_month|monthly_revenue|previous_month_revenue|monthly_growth_percent|
+----------------+-----------------+---------------+----------------------+----------------------+
|            2026|                1|         155000|                  NULL|                  NULL|
|            2026|                2|          39000|                155000|                -74.84|
|            2026|                3|          60000|                 39000|                 53.85|
|            2026|                4|          20500|                 60000|                -65.83|
|            2026|                5|          36000|                 20500|                 75.61|
|            2026|                6|          74000|                 36000|                105.56|
|            2026|                7|          31000|                 74000|         

In [87]:
print("  TOTAL REVENUE  ")

retail_data.agg(
    sum("total_revenue").alias("total_revenue")
).show()

  TOTAL REVENUE  
+-------------+
|total_revenue|
+-------------+
|       468000|
+-------------+



In [88]:
print("  TOTAL ORDERS  ")

retail_data.agg(
    count("transaction_id").alias("total_orders")
).show()

  TOTAL ORDERS  
+------------+
|total_orders|
+------------+
|          16|
+------------+



In [89]:
print("  AVERAGE ORDER VALUE  ")

retail_data.agg(
    round(
        avg("total_revenue"),
        2
    ).alias("average_order_value")
).show()

  AVERAGE ORDER VALUE  
+-------------------+
|average_order_value|
+-------------------+
|            29250.0|
+-------------------+



In [90]:
print("  TOTAL CUSTOMERS  ")

retail_data.select(
    "customer_id"
).distinct().agg(
    count("customer_id").alias("total_customers")
).show()

  TOTAL CUSTOMERS  
+---------------+
|total_customers|
+---------------+
|             10|
+---------------+



In [91]:
active_customers = retail_data.select(
    "customer_id"
).distinct()

print("  ACTIVE CUSTOMERS  ")
print("Active Customers:", active_customers.count())

  ACTIVE CUSTOMERS  
Active Customers: 10


In [92]:
repeat_customers = retail_data.groupBy(
    "customer_id",
    "customer_name"
).agg(
    count("transaction_id").alias("order_count")
).filter(
    col("order_count") > 1
)

print("  REPEAT CUSTOMERS  ")
repeat_customers.show()

print("Repeat Customer Count:", repeat_customers.count())

  REPEAT CUSTOMERS  
+-----------+---------------+-----------+
|customer_id|  customer_name|order_count|
+-----------+---------------+-----------+
|       C105|    Salman Khan|          2|
|       C101|Abhishek Jadhav|          4|
|       C103| Amitabh Sharma|          2|
|       C102|     Tony Stark|          2|
+-----------+---------------+-----------+

Repeat Customer Count: 4


In [93]:
top_10_products = product_revenue.orderBy(
    col("product_revenue").desc()
).limit(10)

print("  TOP 10 PRODUCTS BY REVENUE  ")
top_10_products.show()

  TOP 10 PRODUCTS BY REVENUE  
+----------+-------------+-----------+---------------+
|product_id| product_name|   category|product_revenue|
+----------+-------------+-----------+---------------+
|      P102|   Smartphone|Electronics|         180000|
|      P101|       Laptop|Electronics|         130000|
|      P106|Running Shoes|   Footwear|          35000|
|      P110|  Smart Watch|Electronics|          30000|
|      P103|   Headphones|Electronics|          25000|
|      P104| Office Chair|  Furniture|          24000|
|      P105|  Study Table|  Furniture|          15000|
|      P108|      T-Shirt|   Clothing|          13500|
|      P109|       Jacket|   Clothing|           9000|
|      P107| Sports Shoes|   Footwear|           6500|
+----------+-------------+-----------+---------------+



In [94]:
top_10_customers = customer_revenue.orderBy(
    col("customer_revenue").desc()
).limit(10)

print("  TOP 10 CUSTOMERS BY SALES  ")
top_10_customers.show()

  TOP 10 CUSTOMERS BY SALES  
+-----------+---------------+----------------+
|customer_id|  customer_name|customer_revenue|
+-----------+---------------+----------------+
|       C102|     Tony Stark|          155000|
|       C101|Abhishek Jadhav|          126500|
|       C105|    Salman Khan|           66000|
|       C103| Amitabh Sharma|           60000|
|       C104|     Jaya Mehta|           15000|
|       C106|    Rahul Verma|           14000|
|       C110|    Samay Raina|           10000|
|       C109|     Neha Singh|            9000|
|       C107|     Priya Shah|            6500|
|       C108|   Kapil Sharma|            6000|
+-----------+---------------+----------------+



In [95]:
print("  MONTHLY SALES TREND  ")
monthly_revenue.show()

  MONTHLY SALES TREND  
+----------------+-----------------+---------------+
|transaction_year|transaction_month|monthly_revenue|
+----------------+-----------------+---------------+
|            2026|                1|         155000|
|            2026|                2|          39000|
|            2026|                3|          60000|
|            2026|                4|          20500|
|            2026|                5|          36000|
|            2026|                6|          74000|
|            2026|                7|          31000|
|            2026|                8|          52500|
+----------------+-----------------+---------------+



In [96]:
category_performance = retail_data.groupBy(
    "category"
).agg(
    sum("total_revenue").alias("total_revenue"),
    sum("quantity_sold").alias("total_quantity"),
    count("transaction_id").alias("total_orders")
).orderBy(
    col("total_revenue").desc()
)

print("  CATEGORY PERFORMANCE  ")
category_performance.show()

  CATEGORY PERFORMANCE  
+-----------+-------------+--------------+------------+
|   category|total_revenue|total_quantity|total_orders|
+-----------+-------------+--------------+------------+
|Electronics|       365000|            12|           8|
|   Footwear|        41500|             6|           3|
|  Furniture|        39000|             3|           2|
|   Clothing|        22500|            11|           3|
+-----------+-------------+--------------+------------+



In [97]:
print("  CITY WISE REVENUE  ")
city_sales.show()

  CITY WISE REVENUE  
+---------+------------+
|     city|city_revenue|
+---------+------------+
|   Mumbai|      201500|
|     Pune|      179000|
|    Delhi|       66000|
|Ahmedabad|       21500|
+---------+------------+



In [98]:
best_selling_products = retail_data.groupBy(
    "product_id",
    "product_name"
).agg(
    sum("quantity_sold").alias("total_quantity_sold")
).orderBy(
    col("total_quantity_sold").desc()
)

print("  BEST SELLING PRODUCTS  ")
best_selling_products.show()

  BEST SELLING PRODUCTS  
+----------+-------------+-------------------+
|product_id| product_name|total_quantity_sold|
+----------+-------------+-------------------+
|      P108|      T-Shirt|                  9|
|      P106|Running Shoes|                  5|
|      P103|   Headphones|                  5|
|      P102|   Smartphone|                  4|
|      P109|       Jacket|                  2|
|      P101|       Laptop|                  2|
|      P104| Office Chair|                  2|
|      P110|  Smart Watch|                  1|
|      P107| Sports Shoes|                  1|
|      P105|  Study Table|                  1|
+----------+-------------+-------------------+



In [99]:
from pyspark.sql.functions import row_number

product_window = Window.orderBy(
    col("product_revenue").desc()
)

product_ranking = product_revenue.withColumn(
    "rank",
    row_number().over(product_window)
)

print("  PRODUCT PERFORMANCE RANKING  ")
product_ranking.show()

  PRODUCT PERFORMANCE RANKING  
+----------+-------------+-----------+---------------+----+
|product_id| product_name|   category|product_revenue|rank|
+----------+-------------+-----------+---------------+----+
|      P102|   Smartphone|Electronics|         180000|   1|
|      P101|       Laptop|Electronics|         130000|   2|
|      P106|Running Shoes|   Footwear|          35000|   3|
|      P110|  Smart Watch|Electronics|          30000|   4|
|      P103|   Headphones|Electronics|          25000|   5|
|      P104| Office Chair|  Furniture|          24000|   6|
|      P105|  Study Table|  Furniture|          15000|   7|
|      P108|      T-Shirt|   Clothing|          13500|   8|
|      P109|       Jacket|   Clothing|           9000|   9|
|      P107| Sports Shoes|   Footwear|           6500|  10|
+----------+-------------+-----------+---------------+----+



In [100]:
monthly_revenue = monthly_revenue.cache()

monthly_revenue.count()

print(
    "Aggregated Sales Cached:",
    monthly_revenue.is_cached
)

Aggregated Sales Cached: True


In [101]:
print(
    "Current Partitions:",
    retail_data.rdd.getNumPartitions()
)

Current Partitions: 1


In [102]:
retail_partitioned = retail_data.repartition(4)

print(
    "Partitions After Repartition:",
    retail_partitioned.rdd.getNumPartitions()
)

Partitions After Repartition: 4


In [103]:
print("  EXECUTION PLAN  ")
retail_partitioned.explain()

  EXECUTION PLAN  
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ShuffleQueryStage 5
   +- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=33189]
      +- *(5) Project [transaction_id#19863, customer_id#33609, product_id#33611, quantity_sold#33613, transaction_date#33615, sales_amount#33617, customer_name#20217, gender#20224, age#19751, city#20231, registration_date#19753, product_name#20324, category#20330, brand#20336, unit_price#19813, (quantity_sold#33613 * unit_price#19813) AS total_revenue#21924, year(transaction_date#33615) AS transaction_year#22349, month(transaction_date#33615) AS transaction_month#22367]
         +- *(5) BroadcastHashJoin [product_id#33611], [product_id#19809], Inner, BuildRight, false
            :- *(5) Project [transaction_id#19863, customer_id#33609, product_id#33611, quantity_sold#33613, transaction_date#33615, sales_amount#33617, customer_name#20217, gender#20224, age#19751, city#20231, registration_date#

In [104]:
retail_partitioned.write \
    .mode("overwrite") \
    .parquet("output/cleaned_retail_data")

In [105]:
retail_data.write \
    .mode("overwrite") \
    .partitionBy(
        "transaction_year",
        "transaction_month",
        "category"
    ) \
    .parquet("output/partitioned_retail_data")

In [106]:
category_performance.write \
    .mode("overwrite") \
    .parquet("output/data_mart/category_performance")

In [107]:
top_10_products.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/reports/top_10_products")

top_10_customers.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/reports/top_10_customers")

monthly_revenue.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/reports/monthly_sales")

category_performance.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/reports/category_performance")

city_sales.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/reports/city_revenue")

In [108]:
print("  PIPELINE RESULTS  ")

print(
    "Clean Customers:",
    clean_customers.count()
)

print(
    "Clean Products:",
    clean_products.count()
)

print(
    "Clean Sales:",
    clean_sales.count()
)

print(
    "Combined Retail Records:",
    retail_data.count()
)
print("\nRetail ETL Pipeline Completed.")

  PIPELINE RESULTS  
Clean Customers: 10
Clean Products: 10
Clean Sales: 16
Combined Retail Records: 16

Retail ETL Pipeline Completed.


In [109]:
clean_customers.unpersist()
clean_products.unpersist()
monthly_revenue.unpersist()

print(
    "Customer Cached:",
    clean_customers.is_cached
)

print(
    "Product Cached:",
    clean_products.is_cached
)

print(
    "Monthly Revenue Cached:",
    monthly_revenue.is_cached
)

Customer Cached: False
Product Cached: False
Monthly Revenue Cached: False
